In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
#basemessage is collection of all HumanMessage,AIMessage
from typing import TypedDict, Annotated

from langgraph.checkpoint.memory import MemorySaver #now we are using RAM to store the chat history ***
 

c:\Users\Sachin S\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
       messages:Annotated[list[BaseMessage], add_messages]

In [3]:
model=ChatOpenAI()

def chat_node(state:ChatState):
    #take user query from state
    messages=state['messages']
    #send to llm
    response=model.invoke(messages)
    #response store state
    return {'messages':[response]}

In [4]:
checkpointer=MemorySaver()  #*****

graph=StateGraph(ChatState)
graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)
chatbot=graph.compile(checkpointer=checkpointer)  #****
inital_state={
    'messages':[HumanMessage(content='What is the capital of India')]
}

chatbot.invoke(inital_state)




ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [ ]:
thread_id=1  #**** thread id for each conversation chat session

while True:
    user_message=input('TypeHere:')
    print("User:",user_message)

    if user_message.strip().lower() in ['quit','exit','bye']:
        break

    config={'configurable':{'thread_id':thread_id}} # ***

    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print("AI Message: ",response['messages'][-1].content)

'''Here the problem with the chatbot isis resolved where the  chat history will be maintained'''


User: Hi 
AI Message:  Hello! How can I assist you today?
User: My name is Sachin
AI Message:  Nice to meet you, Sachin! How can I help you today?
User: what is my name?
AI Message:  Your name is Sachin.
User: Can you add 2 and 5 
AI Message:  Of course! 2 + 5 equals 7. Do you have any other math problems you need help with?
User: Now can you multiply the result with 5
AI Message:  Yes, I can do that. 

7 (result of 2 + 5) multiplied by 5 is 35.
User: exit


"Here the problem with the chatbot is ,it doesn't hold any chat history where everytime it will\n invoke with fresh usermessage and every time is a fresh invoke \n so we should use the persistence concept to have the chat history "